In [2]:
import os
import time
import logging
from datetime import datetime, timezone

import requests
import psycopg2
import schedule
from dotenv import load_dotenv
import pandas as pd
import json



In [6]:
# --- CONFIGURATION ---
# Load environment variables from .env file
load_dotenv()

# Set up basic logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Load credentials and settings from environment variables
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")
DB_HOST = os.getenv("DB_HOST") # This will be 'db' from docker-compose

# List of cities to monitor
with open("city.list.json", "r", encoding="utf-8") as read:
    cities = json.load(read)

# Filter cities for Nigeria (country code = 'NG')
nigeria_cities = [c for c in cities if c["country"] == "NG"]

# Extract just the city names into a list
nigeria_city_names = [c["name"] for c in nigeria_cities]

# Print some details
print(f"Total Nigerian cities found: {len(nigeria_city_names)}")
print(nigeria_city_names)  # print first 30 cities for preview


Total Nigerian cities found: 521
['Zuru', 'Zungeru', 'Zaria', 'Zango', 'Zalanga', 'Zaki Biam', 'Zadawa', 'Yuli', 'Yola', 'Yenagoa', 'Yelwa', 'Yashikira', 'Yandev', 'Yanda Bayo', 'Yamrat', 'Yajiwa', 'Wuyo', 'Wusasa', 'Wurno', 'Wukari', 'Wudil', 'Wawa', 'Wauro Jabbe', 'Wasagu', 'Warri', 'Wamba', 'Walmayo', 'Wagini', 'Vom', 'Uyo', 'Uruobo-Okija', 'Uromi', 'Umunede', 'Umuahia', 'Ukata', 'Ughelli', 'Ugep', 'Uga Soja', 'Uga', 'Udi', 'Ubiaja', 'Uba', 'Tungan Ali', 'Toungo', 'Tokombere', 'Tegina', 'Tambuwal', 'Talata Mafara', 'Takum', 'Takai', 'Suya', 'Suleja', 'Sokoto State', 'Sokoto', 'Soba', 'Siluko', 'Shinkafi', 'Shani', 'Shanga', 'Saki', 'Shagamu', 'Shaffa', 'Sauri', 'Sapele', 'Samamiya', 'Sade', 'Runka', 'Ruma', 'Rivers State', 'Riti', 'Ringim', 'Rijau', 'Rano', 'Saminaka', 'Rabah', 'Potiskum', 'Port Harcourt', 'Plateau State', 'Pindiga', 'Patigi', 'Patani', 'Panyam', 'Pankshin', 'Ozubulu', 'Oyo State', 'Oyo', 'Oyan', 'Owode', 'Owo', 'Owerri', 'Otukpa', 'Otta', 'Osogbo', 'Oron', 'Orodo',

In [8]:
pd_cities.to_csv("nigerian_cities.csv", index=False)

In [12]:
import pandas as pd

# === CONFIGURATION ===
input_file = "nigerian_cities.csv"         # Path to your original CSV
output_file = "nigerian_cities.csv"       # Path to save modified CSV
column_to_remove = "state"   # Column you want to remove
old_column_name = "confirmed_state"         # Column to rename
new_column_name = "state"     # New name for that column

# === STEP 1: Read CSV ===
df = pd.read_csv(input_file)

# === STEP 2: Remove a column ===
if column_to_remove in df.columns:
    df = df.drop(columns=[column_to_remove])
else:
    print(f"⚠️ Column '{column_to_remove}' not found — skipping removal.")

# === STEP 3: Rename a column ===
if old_column_name in df.columns:
    df = df.rename(columns={old_column_name: new_column_name})
else:
    print(f"⚠️ Column '{old_column_name}' not found — skipping rename.")

# === STEP 4: Save the updated CSV ===
df.to_csv(output_file, index=False)

print(f"✅ CSV updated and saved to '{output_file}'")


✅ CSV updated and saved to 'nigerian_cities.csv'


In [3]:
def fetch_weather(city: str) -> dict | None:
    """Fetches weather data for a given city from the OpenWeatherMap API."""
    api_url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={OPENWEATHER_API_KEY}"
    try:
        response = requests.get(api_url, timeout=10)
        response.raise_for_status()  # Raises an HTTPError for bad responses (4xx or 5xx)
        return response.json()
    except requests.exceptions.RequestException as e:
        logging.error(f"API request failed for {city}: {e}")
        return None

def process_weather_data(city: str, data: dict) -> dict | None:
    """Processes raw JSON data into a structured dictionary."""
    if not data:
        return None
    try:
        processed = {
            "city_name": city,
            # Convert temperature from Kelvin to Celsius
            "temperature": round(data['main']['temp'] - 273.15, 2),
            "humidity": data['main']['humidity'],
            "pressure": data['main']['pressure'],
            "wind_speed": data['wind']['speed'],
            "weather_main": data['weather'][0]['main'],
            "weather_desc": data['weather'][0]['description'],
            # Convert Unix timestamp to a timezone-aware datetime object
            "reading_timestamp": datetime.fromtimestamp(data['dt'], tz=timezone.utc)
        }
        return processed
    except (KeyError, IndexError) as e:
        logging.error(f"Error processing data for {city}. Missing key: {e}")
        return None


In [4]:
import logging
import pandas as pd
import os

def weather_pipeline_job():
    """Main ETL process for all Nigerian cities."""
    logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
    logging.info("Starting weather data pipeline job...")

    # Ensure data folder exists
    os.makedirs("./data", exist_ok=True)

    all_raw_data = []
    all_processed_data = []

    for city in nigeria_city_names:
        logging.info(f"Fetching weather data for {city}...")

        try:
            raw_data = fetch_weather(city)

            # Skip empty or failed responses
            if not raw_data:
                logging.warning(f"No data returned for {city}, skipping.")
                continue

            # Normalize raw API data safely
            pd_raw_data = pd.json_normalize(raw_data)
            pd_raw_data["city"] = city
            all_raw_data.append(pd_raw_data)

            # Processed data (assumes your function returns dict or list of dicts)
            processed_data = process_weather_data(city, raw_data)
            pd_processed_data = pd.DataFrame([processed_data])
            pd_processed_data["city"] = city
            all_processed_data.append(pd_processed_data)

        except Exception as e:
            logging.error(f"Error processing {city}: {e}")

    # Combine and save once after the loop
    if all_raw_data:
        df_raw = pd.concat(all_raw_data, ignore_index=True)
        df_raw.to_csv("./data/raw.csv", index=False)
        logging.info(f"Saved raw data for {len(df_raw)} cities.")

    if all_processed_data:
        df_processed = pd.concat(all_processed_data, ignore_index=True)
        df_processed.to_csv("./data/processed.csv", index=False)
        logging.info(f"Saved processed data for {len(df_processed)} cities.")

    logging.info("Weather data pipeline job finished.")



In [5]:
weather_pipeline_job()

2025-10-17 12:33:19,779 - INFO - Starting weather data pipeline job...
2025-10-17 12:33:19,782 - INFO - Fetching weather data for Zuru...
2025-10-17 12:33:20,810 - INFO - Fetching weather data for Zungeru...
2025-10-17 12:33:21,520 - INFO - Fetching weather data for Zaria...
2025-10-17 12:33:22,055 - INFO - Fetching weather data for Zango...
2025-10-17 12:33:22,474 - INFO - Fetching weather data for Zalanga...
2025-10-17 12:33:23,043 - INFO - Fetching weather data for Zaki Biam...
2025-10-17 12:33:24,346 - INFO - Fetching weather data for Zadawa...
2025-10-17 12:33:25,228 - INFO - Fetching weather data for Yuli...
2025-10-17 12:33:25,823 - INFO - Fetching weather data for Yola...
2025-10-17 12:33:26,146 - INFO - Fetching weather data for Yenagoa...
2025-10-17 12:33:26,792 - INFO - Fetching weather data for Yelwa...
2025-10-17 12:33:27,412 - INFO - Fetching weather data for Yashikira...
2025-10-17 12:33:28,195 - INFO - Fetching weather data for Yandev...
2025-10-17 12:33:28,557 - INFO -